# 两个 2026 年 7 月冲击日对静态组成 126 日窗口的直接覆盖上界

只读固定 Stage1 静态组成 0bp 自有账户及全部 819 个 Qlib Recorder 滚动窗口。即使完美翻正每一个被两天直接影响的负窗口，也只是固定其余收益路径的乐观计数上界；真实策略未来持仓与风险状态可能改变，不能把此上界当成策略不可能性证明。

In [1]:
import os,json,hashlib,math
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_static_composition_tail_window_bound_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda x:hashlib.sha256(Path(x).read_bytes()).hexdigest()
source=root/plan['sources']['static_composition_card'];tail=root/plan['sources']['selection_tail_result']
assert sha(source/'summary.json')==plan['sources']['static_composition_summary_sha256'] and sha(tail)==plan['sources']['selection_tail_result_sha256']
summary=json.loads((source/'summary.json').read_text());tail_result=json.loads(tail.read_text())
assert summary['recorder_id']==plan['sources']['static_composition_recorder_id'] and summary['rolling_summary']['windows']==819
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['sources']['provider_uri']),region='us',kernels=1)
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
 book=rec.load_object('native/candidate_0');rolling=rec.load_object('analysis/rolling/all_windows')
a=book['authority'];assert len(a)==944 and len(rolling)==819 and pd.DatetimeIndex(a.trade_date).is_unique
assert all(pd.Timestamp(rolling.first_trade.iloc[j])==pd.Timestamp(a.trade_date.iloc[j]) and pd.Timestamp(rolling.last_exit.iloc[j])==pd.Timestamp(a.exit_date.iloc[j+125]) for j in range(819))
q3=tail_result['Q3_2026_negative_days'];worst=sorted(q3,key=lambda x:x['SOXL_selection_log'])[:2]
assert set(pd.Timestamp(x['trade_date']).strftime('%Y-%m-%d') for x in worst)==set(plan['sources']['fixed_trade_dates'])
dates=pd.DatetimeIndex(a.trade_date).strftime('%Y-%m-%d').tolist();idx=[dates.index(x) for x in plan['sources']['fixed_trade_dates']]
affected=np.array([any(j<=i<=j+125 for i in idx) for j in range(819)],dtype=bool)
positive=rolling.relative_log.to_numpy(float)>0
assert np.isfinite(rolling.relative_log.to_numpy(float)).all() and int(positive.sum())==round(summary['rolling_summary']['positive_fraction']*819)
required=math.ceil(.70*819);flippable=int((affected & ~positive).sum());optimistic=int(positive.sum())+flippable
result={'status':'two_July_tail_days_fixed_path_window_coverage_cannot_reach_70pct','plan_sha256':sha(card/'plan.json'),'source_recorder_id':summary['recorder_id'],'source_NAS_sha256':summary['nas_evidence']['sha256'],'sessions':944,'rolling126_windows':819,'fixed_trade_dates':plan['sources']['fixed_trade_dates'],'fixed_trade_indices':idx,'existing_positive_windows':int(positive.sum()),'required_positive_windows_for_70pct':required,'directly_affected_windows':int(affected.sum()),'directly_affected_negative_windows':flippable,'optimistic_max_positive_windows':optimistic,'optimistic_max_positive_fraction':optimistic/819,'optimistic_shortfall_windows':max(0,required-optimistic),'full_strategy_with_state_feedback_ruled_out':False,'source_option_skew_value_ruled_out':False,'new_market_requests':0,'model_fits':0,'new_native_accounts':0,'bootstrap_runs':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ['existing_positive_windows','required_positive_windows_for_70pct','directly_affected_windows','directly_affected_negative_windows','optimistic_max_positive_windows','optimistic_shortfall_windows']})

{'existing_positive_windows': 458, 'required_positive_windows_for_70pct': 574, 'directly_affected_windows': 68, 'directly_affected_negative_windows': 34, 'optimistic_max_positive_windows': 492, 'optimistic_shortfall_windows': 82}


## 中文结论

静态 Stage1 组成账户原有 458/819 个正的 126 日 SOXX 超额窗口，70% 门需 574 个。2026-07-06 与 07-27 只能直接影响 68 个窗口，其中 34 个目前为负；即使全部不可能地翻正，其余日收益冻结时最多 492/819（60.07%），还差 82 个。此为固定路径直接日效应的乐观上界，真实自有状态策略可因后续持仓/费用反馈影响更多日，不能据此宣布任何策略不可能。0 新拟合、账户或 bootstrap；证据见 `result.json`。